# Mythic+ Gold Layer

This notebook builds the Gold layer for the Mythic+ pipeline. The Gold layer organizes data into fact and dimension tables to support reporting and analysis.

## Fact Tables

**fact_mythic_runs**
Contains one row per player per run. This is the main table used for performance analysis.

* keystone_run_id
* character_name
* class
* spec
* role
* dungeon_id
* keystone_level
* clear_time_ms
* completed_at
* score

**fact_boss_encounters**
Contains one row per boss encounter within a run.

* keystone_run_id
* encounter_id
* boss_name
* duration_ms
* is_success
* dungeon_id
* keystone_level
* completed_at

## Dimension Tables

**dim_dungeons**
Provides static information about each dungeon.

* dungeon_id
* dungeon_name
* expansion_id
* timer_ms

**dim_characters**
Provides descriptive information about each character.

* character_name
* class
* spec
* role
* realm
* region

**dim_bosses** (optional)
Provides a lookup for boss metadata.

* encounter_id
* boss_name

The fact tables capture events, while the dimension tables provide context for analysis.


In [0]:
players_df = spark.table("workspace.silver_mythic_plus.players")
dungeons_df = spark.table("workspace.silver_mythic_plus.dungeons")
boss_df = spark.table("workspace.silver_mythic_plus.boss_encounters")

In [0]:
from pyspark.sql.functions import col

fact_runs_df = (
    players_df.alias("p")
    .join(
        dungeons_df.alias("d"),
        col("p.keystone_run_id") == col("d.keystone_run_id"),
        "inner"
    )
    .select(
        col("p.keystone_run_id"),
        col("p.character_name"),
        col("p.class"),
        col("p.spec"),
        col("p.role"),

        col("d.dungeon_id"),
        col("d.dungeon_name"),
        col("d.keystone_level"),
        col("d.clear_time_ms"),
        col("d.completed_at"),
        col("d.score"),
        col("d.season")
    )
)

In [0]:
dim_dungeons_df = (
    dungeons_df
    .select(
        "dungeon_id",
        "dungeon_name",
        "expansion_id",
        "timer_ms"
    )
    .dropDuplicates()
)

In [0]:
dim_characters_df = (
    players_df
    .select(
        "character_name",
        "class",
        "spec",
        "role",
        "realm",
        "region"
    )
    .dropDuplicates()
)

In [0]:
fact_boss_df = (
    boss_df.alias("b")
    .join(
        dungeons_df.alias("d"),
        col("b.keystone_run_id") == col("d.keystone_run_id"),
        "left"
    )
    .select(
        col("b.keystone_run_id"),
        col("b.encounter_id"),
        col("b.boss_name"),
        col("b.fight_duration_ms"),
        col("b.boss_killed"),

        col("d.dungeon_id"),
        col("d.keystone_level"),
        col("d.completed_at")
    )
)

display(fact_boss_df)

In [0]:
dim_bosses_df = (
    boss_df
    .select("encounter_id", "boss_name")
    .dropDuplicates()
)

In [0]:
fact_runs_df.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .saveAsTable("workspace.gold_mythic_plus.fact_mythic_runs")

dim_dungeons_df.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .saveAsTable("workspace.gold_mythic_plus.dim_dungeons")

dim_characters_df.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .saveAsTable("workspace.gold_mythic_plus.dim_characters")

fact_boss_df.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .saveAsTable("workspace.gold_mythic_plus.fact_boss_encounters")

dim_bosses_df.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .saveAsTable("workspace.gold_mythic_plus.dim_bosses")